In [5]:
from dotenv import load_dotenv
import os 
from google import genai

load_dotenv()
api_key = os.getenv("API_KEY")
client = genai.Client(api_key=api_key)
response = client.models.generate_content(
    model = "gemini-2.5-flash", contents="Tell me a programming joke"
)

response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text="""Why do programmers prefer dark mode?

Because light attracts bugs!"""
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-flash',
  response_id='fJ2-aKjLMoe_vdIP6LSAoQw',
  sdk_http_response=HttpResponse(
    headers=<dict len=11>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=13,
    prompt_token_count=6,
    prompt_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.TEXT: 'TEXT'>,
        token_count=6
      ),
    ],
    thoughts_token_count=204,
    total_token_count=223
  )
)

In [6]:
response.text

'Why do programmers prefer dark mode?\n\nBecause light attracts bugs!'

In [7]:

print(response.text)

Why do programmers prefer dark mode?

Because light attracts bugs!


In [8]:
def ask_llm(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

ask_llm("Du är en Göteborgare, ge mig ett skämt som är go")

'Hallå där, goa gubbe! Eller tjej! Klart du ska ha ett skämt som är gött! Här kommer ett du kan dra på nästa fika:\n\n***\n\nVarför är alla spårvagnar i Göteborg så pratglada?\n\n... För de har ju *spår* av gamla historier! Änna!\n\n***\n\nHaha, hoppas den var lite go! Du får ha det så bra, la!'

In [9]:
response = ask_llm("""
    Du är en expert inom köp och sälj av bostäder, likt en proffsig mäklare.
    Generera bostadspriser, månadsavgifter, address, stad, boarea i jsonformat (ej markdown)

    Exempel:
            {
                "address": "Fågelvägen 5,
                "price_sek": 3000000,
                "city": "Göteborg",
                "monthly_fee": 4000,
                "area": 60
            }   
                   
    Ge mig en lista på 5 bostäder
""")

response

'[\n    {\n        "address": "Karlbergsvägen 77",\n        "price_sek": 6500000,\n        "city": "Stockholm",\n        "monthly_fee": 3800,\n        "area": 65\n    },\n    {\n        "address": "Linnégatan 24",\n        "price_sek": 4200000,\n        "city": "Göteborg",\n        "monthly_fee": 4500,\n        "area": 70\n    },\n    {\n        "address": "Davidshallsgatan 12",\n        "price_sek": 3100000,\n        "city": "Malmö",\n        "monthly_fee": 3200,\n        "area": 60\n    },\n    {\n        "address": "Dragarbrunnsgatan 50",\n        "price_sek": 2750000,\n        "city": "Uppsala",\n        "monthly_fee": 2900,\n        "area": 55\n    },\n    {\n        "address": "Starrgränd 3",\n        "price_sek": 3900000,\n        "city": "Stockholm",\n        "monthly_fee": 4700,\n        "area": 75\n    }\n]'

In [10]:
print(response)

[
    {
        "address": "Karlbergsvägen 77",
        "price_sek": 6500000,
        "city": "Stockholm",
        "monthly_fee": 3800,
        "area": 65
    },
    {
        "address": "Linnégatan 24",
        "price_sek": 4200000,
        "city": "Göteborg",
        "monthly_fee": 4500,
        "area": 70
    },
    {
        "address": "Davidshallsgatan 12",
        "price_sek": 3100000,
        "city": "Malmö",
        "monthly_fee": 3200,
        "area": 60
    },
    {
        "address": "Dragarbrunnsgatan 50",
        "price_sek": 2750000,
        "city": "Uppsala",
        "monthly_fee": 2900,
        "area": 55
    },
    {
        "address": "Starrgränd 3",
        "price_sek": 3900000,
        "city": "Stockholm",
        "monthly_fee": 4700,
        "area": 75
    }
]


In [11]:
from pydantic import BaseModel, Field
import json 

class Apartment(BaseModel):
    address: str 
    city: str 
    price_sek: int = Field(gt=1000000, lt = 8000000) 
    monthly_fee: int 
    area: int 

class ApartmentList(BaseModel):
    objects: list[Apartment]


apartments = ApartmentList.model_validate({"objects": json.loads(response)})
apartments
    

ApartmentList(objects=[Apartment(address='Karlbergsvägen 77', city='Stockholm', price_sek=6500000, monthly_fee=3800, area=65), Apartment(address='Linnégatan 24', city='Göteborg', price_sek=4200000, monthly_fee=4500, area=70), Apartment(address='Davidshallsgatan 12', city='Malmö', price_sek=3100000, monthly_fee=3200, area=60), Apartment(address='Dragarbrunnsgatan 50', city='Uppsala', price_sek=2750000, monthly_fee=2900, area=55), Apartment(address='Starrgränd 3', city='Stockholm', price_sek=3900000, monthly_fee=4700, area=75)])

In [12]:

apartments.objects

[Apartment(address='Karlbergsvägen 77', city='Stockholm', price_sek=6500000, monthly_fee=3800, area=65),
 Apartment(address='Linnégatan 24', city='Göteborg', price_sek=4200000, monthly_fee=4500, area=70),
 Apartment(address='Davidshallsgatan 12', city='Malmö', price_sek=3100000, monthly_fee=3200, area=60),
 Apartment(address='Dragarbrunnsgatan 50', city='Uppsala', price_sek=2750000, monthly_fee=2900, area=55),
 Apartment(address='Starrgränd 3', city='Stockholm', price_sek=3900000, monthly_fee=4700, area=75)]

In [13]:
apartments.objects[1].address, apartments.objects[1].city

('Linnégatan 24', 'Göteborg')

In [ ]:
addresses = [
    [home.address, home.city, home.price_sek, home.monthly_fee]
    for home in apartments.objects
    if   4000000 < home.price_sek < 8000000
]

addresses

[['Karlbergsvägen 77', 'Stockholm', 6500000, 3800],
 ['Linnégatan 24', 'Göteborg', 4200000, 4500]]

In [36]:
import pandas as pd

filtered_homes = [
    home for home in apartments.objects if 4_000_000 < home.price_sek < 8_000_000
]
# Convert to df
df = pd.DataFrame(
    [home.model_dump(include={"address", "city", "price_sek", "monthly_fee"}) for home in filtered_homes]
)
df

,address,city,price_sek,monthly_fee
0,Karlbergsvägen 77,Stockholm,6500000,3800
1,Linnégatan 24,Göteborg,4200000,4500


In [37]:
df

,address,city,price_sek,monthly_fee
0,Karlbergsvägen 77,Stockholm,6500000,3800
1,Linnégatan 24,Göteborg,4200000,4500


In [43]:
import duckdb
import dlt

@dlt.resource(write_disposition="replace", table_name="apartment")
def load_data():
    for record in df.to_dict(orient="records"):
        yield record
        
pipeline = dlt.pipeline(
    pipeline_name="apartments",
    destination= "duckdb",
    dataset_name="staging",
)

load_info = pipeline.run(load_data())
print(load_info)

Pipeline apartments load step completed in 0.13 seconds
1 load package(s) were loaded to destination duckdb and into dataset staging
The duckdb destination used duckdb:///c:\Users\salih\OneDrive\Dokument\Github\ai_engineering_salih_morina\code-alongs\07_pydantic_geminai\apartments.duckdb location to store data
Load package 1757331598.140868 is LOADED and contains no failed jobs
